# ReplenishIQ - multi-agent replenishment copilot (demo)

A LangGraph **supervisor** coordinates four workers:

| Agent | What it does | LLM? |
|---|---|---|
| inventory_agent | stock + supplier terms via **MCP tools** (Streamable HTTP) | no |
| demand_agent | per-SKU forecast (pandas exponential smoothing) | no |
| policy_agent | **RAG** over policy docs, LLM extracts constraints, code checks grounding | yes (1 call) |
| optimizer_agent | **PuLP MILP** picks order quantities | no |

Then a **human approval** interrupt, the approved PO is written through MCP, and the LLM writes a short explanation.
About 3 Groq calls per run, which fits the free tier.

## 1. Setup
Clone the repo (on Colab) and install the requirements. Pip may warn about conflicts with Colab's preinstalled packages; you can ignore those warnings.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/rkoush/inventory-replenishment-multi-agent-copilot.git"
IN_COLAB = Path("/content").exists()
if IN_COLAB:
    REPO = Path("/content/inventory-replenishment-multi-agent-copilot")
    if not REPO.exists():
        !git clone -q {REPO_URL} {REPO}
else:  # running locally from notebooks/
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repo:", REPO)

In [ ]:
!pip install -q -r requirements.txt

## 2. Groq API key
Add `GROQ_API_KEY` in Colab's **Secrets** panel (key icon on the left) and turn on notebook access.
If there's no key, the notebook runs **offline**: rule-based stand-ins replace the 3 LLM steps, and everything else stays the same.
The cell makes **1 tiny smoke-test call** to show whether Groq is reachable with the chosen model.

In [ ]:
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY") or ""
except Exception as e:  # not in Colab, or the secret is missing
    print("No Colab secret:", type(e).__name__)
os.environ.setdefault("GROQ_MODEL", "openai/gpt-oss-120b")
# os.environ["GROQ_ROUTER_MODEL"] = "openai/gpt-oss-20b"   # routing model (this is already the default)
# Set models BEFORE this import: config reads the env once.
from agents.llm import smoke_test
import config

OFFLINE = not os.environ.get("GROQ_API_KEY")
if OFFLINE:
    print("Mode: OFFLINE (rule-based stand-ins for the 3 LLM steps)")
else:
    ok, msg = smoke_test()  # 1 tiny call: "Reply with OK"
    print(("OK   " if ok else "FAIL ") + msg)
    if ok:
        print(f"Mode: Groq  main={config.GROQ_MODEL}  router={config.GROQ_ROUTER_MODEL}")
    else:
        OFFLINE = True  # don't waste calls on retries/fallbacks; fix the key/model and re-run this cell
        print("Mode: OFFLINE (Groq not reachable with this key/model; see message above)")

## 3. Synthetic retail data
This creates 10 SKUs at 2 stores with 180 days of sales (weekly seasonality and noise), supplier terms, and current stock. The data is seeded, so every run is the same.

In [ ]:
!python -m scripts.generate_data
import sqlite3, pandas as pd
con = sqlite3.connect("data/replenishiq.db")
pd.read_sql("SELECT * FROM inventory i JOIN suppliers s USING (sku) WHERE store='S001'", con)

## 4. RAG index over policy documents
Each markdown section becomes one chunk. Embeddings come from Chroma's local all-MiniLM-L6-v2 model, so no paid API is needed.
Re-running this cell is safe: the index is updated in place, and the retriever reloads it when it changes.


In [ ]:
from rag.build_index import sync_index
from rag.retriever import retrieve
n, status = sync_index()  # in-process; safe to re-run (upserts, keeps the collection)
print(f"Indexed {n} policy chunks ({status})")
for hit in retrieve("safety stock days by category", k=2):
    print(hit["score"], hit["source"], ">", hit["section"])


## 5. Start the MCP server (Streamable HTTP on :8000/mcp)
The server runs as a separate process. Agents find its tools at runtime through `langchain-mcp-adapters`.

In [ ]:
import os, subprocess, time, httpx

def wait_healthy(port, log):
    for _ in range(60):
        try:
            r = httpx.get(f"http://localhost:{port}/health", timeout=1)
            if r.status_code == 200:
                return print(f"server on :{port} up:", r.json())
        except httpx.HTTPError:
            time.sleep(0.5)
    print(open(log).read())

server = subprocess.Popen([sys.executable, "-m", "mcp_servers.inventory_server"],
                          stdout=open("mcp_server.log", "w"), stderr=subprocess.STDOUT)
wait_healthy(8000, "mcp_server.log")

In [ ]:
from agents.mcp_client import mcp_session
async with mcp_session() as tools:
    print("Tools:", tools.names)
    print(await tools.call("get_stock", sku="GRO-001", store="S001"))

## 6. Build the supervisor graph

In [ ]:
from agents.graph import build_graph
from agents.llm import get_llm
import config
llm = get_llm(offline=OFFLINE)
graph = build_graph(llm=llm, router_llm=get_llm(offline=OFFLINE, model=config.GROQ_ROUTER_MODEL))
print(graph.get_graph().draw_mermaid())

## 7. Run until the human-approval interrupt
Each line below is one agent step. The LangGraph checkpointer (MemorySaver) keeps the state while the graph is paused.

In [ ]:
from app.run_demo import stream, print_interrupt
cfg = {"configurable": {"thread_id": "colab-demo-1"}}
request = "Plan next week's replenishment for store S001 with a $5,000 budget"
payload = await stream(graph, {"request": request}, cfg)
print_interrupt(payload)

## 8. Human in the loop
Optional: send the plan back with a smaller budget. The optimizer runs again and the graph pauses again.

In [ ]:
from langgraph.types import Command
payload = await stream(graph, Command(resume={"decision": "revise", "budget": 4000, "approver": "rishi"}), cfg)
print_interrupt(payload)

Approve the plan. `place_order` then calls the MCP `create_purchase_order` tool, and the report node explains the plan.

In [ ]:
payload = await stream(graph, Command(resume={"decision": "approve", "approver": "rishi", "comment": "demo"}), cfg)
from IPython.display import Markdown, display
from agents.report import escape_dollars  # so $ amounts aren't rendered as LaTeX
final = (await graph.aget_state(cfg)).values
display(Markdown(escape_dollars(final["report"])))

In [ ]:
pd.read_sql("SELECT * FROM purchase_orders ORDER BY created_at DESC", con)

## 9. Authentication & authorization: JWT roles and scopes
A second MCP server runs on :8001 with `MCP_AUTH=jwt`. `scripts/mint_tokens.py` stands in for the identity provider: it creates an RSA keypair and one signed token per role in the gitignored `.secrets/` folder.

| Role | Scopes | Can call |
|---|---|---|
| planner | `inventory:read` | stock, sales, and supplier reads |
| approver | `inventory:read po:write` | reads + `create_purchase_order` |
| auditor | `po:read` | `list_purchase_orders` |

The server hides tools a caller has no scope for, and rejects calls to them with a clear `forbidden` error.

In [ ]:
!python -m scripts.mint_tokens --hours 2
from config import role_token
TOKENS = {r: role_token(r) for r in ("planner", "approver", "auditor")}
jwt_server = subprocess.Popen([sys.executable, "-m", "mcp_servers.inventory_server"],
                              env={**os.environ, "MCP_AUTH": "jwt", "MCP_PORT": "8001"},
                              stdout=open("mcp_jwt.log", "w"), stderr=subprocess.STDOUT)
wait_healthy(8001, "mcp_jwt.log")
JWT_URL = "http://localhost:8001/mcp"

In [ ]:
from fastmcp import Client

async def as_role(role, tool, **args):
    try:
        async with Client(JWT_URL, auth=TOKENS[role]) as c:
            out = (await c.call_tool(tool, args)).data
        print(f"{role:>8} -> {tool}: OK  {out}")
    except Exception as e:
        print(f"{role:>8} -> {tool}: {type(e).__name__}: {e}")

for role, tok in TOKENS.items():
    async with Client(JWT_URL, auth=tok) as c:
        print(f"{role:>8} sees:", sorted(t.name for t in await c.list_tools()))
print()
lines = [{"sku": l["sku"], "quantity": l["order_qty"]} for l in final["order_plan"]["lines"] if l["order_qty"] > 0]
await as_role("planner", "create_purchase_order", store="S001", lines=lines, approved_by="planner-bot")
await as_role("auditor", "get_stock", sku="GRO-001", store="S001")

The approver token works, but the agent only uses it in `place_order`, after a human approves at the interrupt. This cell places the plan approved in section 8 with the approver token. It's a demo database, so this simply adds a second PO. Then the auditor reads the PO log.

In [ ]:
await as_role("approver", "create_purchase_order", store="S001", lines=lines,
              approved_by=final["approval"]["approver"], note="approved at the HITL interrupt (JWT demo)")
async with Client(JWT_URL, auth=TOKENS["auditor"]) as c:
    display(pd.DataFrame((await c.call_tool("list_purchase_orders", {"store": "S001", "limit": 5})).data["orders"]))

## 10. Tests (optional)
The test suite starts its own MCP servers on free ports, so it doesn't touch the servers above. It runs before the chat UI so that, after **Run all**, the UI link is the last output. Set `RUN_TESTS = False` to skip it.


In [ ]:
RUN_TESTS = True
if RUN_TESTS:
    !python -m pytest -q -W ignore


## 11. Chat UI (Chainlit)
The same graph as a chat app: each agent step is a collapsible step, the plan appears as a table with **Approve / Reject / Revise** buttons, and the final message shows the PO id and which MCP role token each step used. The UI talks to the JWT server from section 9 (planner token for reads, approver token for the PO).

With a Groq key, every request you type costs the usual ~3 LLM calls.

**Colab caveats.** Colab's port proxy doesn't reliably forward WebSockets, so the app runs with `CHAINLIT_TRANSPORTS=polling`, which falls back to HTTP long-polling and works through the proxy. If the page loads but the chat never connects, use the optional cloudflared cell below.

In [ ]:
!pip install -q -r requirements-ui.txt

In [ ]:
ui_env = {**os.environ, "MCP_AUTH": "jwt", "MCP_SERVER_URL": "http://localhost:8001/mcp",
          "RIQ_OFFLINE": "1" if OFFLINE else "0", "CHAINLIT_TRANSPORTS": "polling"}
ui_proc = subprocess.Popen([sys.executable, "-m", "chainlit", "run", "ui/chainlit_app.py", "--port", "8501", "--headless"],
                           env=ui_env, stdout=open("chainlit.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        if httpx.get("http://localhost:8501/", timeout=1).status_code == 200:
            print("Chainlit up on :8501"); break
    except httpx.HTTPError:
        time.sleep(0.5)
else:
    print(open("chainlit.log").read())

**Optional fallback: Cloudflare quick tunnel** (no account needed). It gives the app a temporary public `https://*.trycloudflare.com` URL with working WebSockets. Anyone with the link can use the app, including placing demo POs, so stop it when you're done (section 12). Set `RUN_TUNNEL = True` to use it.

In [ ]:
RUN_TUNNEL = False
if RUN_TUNNEL:
    import re
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
    tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
                              stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):  # wait until the tunnel is registered, else the URL returns 530 for a while
        log = open("cloudflared.log").read()
        if "Registered tunnel connection" in log:
            PUBLIC_URL = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log).group()
            print("Public URL:", PUBLIC_URL); break
        time.sleep(1)
    else:
        print(log[-2000:])

In [ ]:
# Keep this the last cell that prints: after "Run all", the app link is the last output.
if "PUBLIC_URL" in globals():
    print("Public tunnel URL:", PUBLIC_URL)
if IN_COLAB:
    from google.colab import output
    from google.colab.output import eval_js
    print("Open the chat UI:", eval_js("google.colab.kernel.proxyPort(8501)"))  # link to the proxied app
    output.serve_kernel_port_as_window(8501)  # opens a tab; browsers may block it
else:
    print("Open the chat UI: http://localhost:8501")


**Open the link above** to use the chat UI. If no tab opens (browsers often block it), click the printed link, or use `serve_kernel_port_as_iframe` to embed the app here: set `SHOW_IFRAME = True` in the next cell and run it.

The servers keep running after **Run all**; they stop when the runtime stops, or with the cleanup cell in section 12.


In [ ]:
SHOW_IFRAME = False  # True: embed the app in this notebook
if SHOW_IFRAME and IN_COLAB:
    output.serve_kernel_port_as_iframe(8501, height=800)


## 12. Cleanup (optional)
Stops the MCP servers, the chat UI, and the tunnel. It does nothing by default, so **Run all** leaves the app running. Set `STOP_SERVERS = True` and run the cell when you're done.


In [ ]:
STOP_SERVERS = False  # True: stop everything started above
if STOP_SERVERS:
    for name in ("server", "jwt_server", "ui_proc", "tunnel"):
        if name in globals():
            globals()[name].terminate()
    print("servers stopped")
